# Función de costo en regresión logística

Cuando el pronóstico del clima dice "70% de probabilidad de lluvia", no te está
diciendo si va a llover o no: te está dando un **número entre 0 y 1** que
representa qué tan seguro está. La regresión logística hace exactamente eso,
pero para clasificación binaria: en vez de predecir una cantidad (como en
regresión lineal), predice la **probabilidad** de que una observación
pertenezca a la clase positiva (por ejemplo, "es correo spam" vs. "no lo es").

En este notebook responderemos: **¿cómo convertimos un puntaje cualquiera en
una probabilidad creíble, y cómo medimos qué tan buenas son esas
probabilidades?**

In [1]:
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## 1. Por qué no basta con una recta

En regresión lineal, el modelo calcula un puntaje $z = w^T x + b$ y lo usa
directamente como predicción. Si intentáramos usar ese mismo puntaje como si
fuera una probabilidad, tendríamos un problema: una recta no tiene techo ni
piso, así que para valores de $x$ suficientemente grandes o pequeños, "la
probabilidad" que prediría se saldría del rango $[0,1]$ — algo que no tiene
sentido (¿una probabilidad de 140%? ¿de -30%?).

Veámoslo directamente: imagina que intentamos predecir "¿aprobó el examen?"
(0 = no, 1 = sí) a partir de horas de estudio, usando una recta común.

In [2]:
horas = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8])
aprobo = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1])

# Una recta ajustada "a ojo" para pasar cerca de los puntos.
probabilidad_ingenua = -0.15 + 0.14 * horas

fig = go.Figure()
fig.add_trace(go.Scatter(x=horas, y=aprobo, mode="markers", name="dato real (0 o 1)", marker={"size": 11, "color": "black"}))
fig.add_trace(go.Scatter(x=horas, y=probabilidad_ingenua, mode="lines+markers", name="recta usada como 'probabilidad'", line={"color": "#d62728"}))
fig.add_hrect(y0=1, y1=probabilidad_ingenua.max() + 0.1, fillcolor="red", opacity=0.08, line_width=0, annotation_text="imposible: > 100%")
fig.add_hrect(y0=probabilidad_ingenua.min() - 0.1, y1=0, fillcolor="red", opacity=0.08, line_width=0, annotation_text="imposible: < 0%")
fig.update_layout(title="Una recta común se sale del rango [0, 1]: no sirve como probabilidad", xaxis_title="horas de estudio", yaxis_title="'probabilidad' predicha")
fig.show()

Necesitamos una función que tome cualquier número real (el puntaje $z$, que
puede ser gigante, negativo, lo que sea) y lo "aplaste" para que siempre quede
entre 0 y 1, sin perder el orden (un $z$ más alto debe seguir dando una
probabilidad más alta). La función que hace justo eso es la **sigmoide**:

$$p = \sigma(z) = \frac{1}{1 + e^{-z}}$$

- Si $z$ es muy negativo, $e^{-z}$ se vuelve enorme, así que $p$ se acerca a 0.
- Si $z$ es muy positivo, $e^{-z}$ se acerca a 0, así que $p$ se acerca a 1.
- Si $z=0$, $p = \frac{1}{1+1} = 0.5$: el punto de máxima incertidumbre.

In [3]:
z = np.linspace(-8, 8, 200)
fig = px.line(
    pl.DataFrame({"z": z, "probabilidad": sigmoid(z)}),
    x="z", y="probabilidad",
    title="La sigmoide comprime cualquier puntaje z en una probabilidad entre 0 y 1",
)
fig.add_hline(y=0.5, line_dash="dash", annotation_text="z=0 → p=0.5")
fig.add_hrect(y0=0, y1=1, fillcolor="green", opacity=0.05, line_width=0)
fig.show()

Por tanto, el modelo completo es $p = \sigma(w^T x + b) = P(y=1 \mid x)$, y
siempre queda entre 0 y 1. La etiqueta real $y$ vale 0 o 1 (nunca "40%
verdadero"): es la sigmoide la que convierte el puntaje en una probabilidad
continua.

## 2. ¿Qué tan bien predijo el modelo la etiqueta real?

Antes de escribir ninguna fórmula general, veamos casos concretos. Supón que
para 4 observaciones distintas (todas con $y=1$, es decir, la clase positiva
sí ocurrió) el modelo predijo estas probabilidades:

In [4]:
casos = pl.DataFrame({
    "caso": ["muy seguro y acertó", "algo seguro y acertó", "algo seguro y falló", "muy seguro y falló"],
    "y_real": [1, 1, 1, 1],
    "p_predicha": [0.99, 0.60, 0.40, 0.01],
})
casos

caso,y_real,p_predicha
str,i64,f64
"""muy seguro y acertó""",1,0.99
"""algo seguro y acertó""",1,0.6
"""algo seguro y falló""",1,0.4
"""muy seguro y falló""",1,0.01


Intuitivamente: el primer caso debería tener un costo bajísimo (el modelo
estaba casi seguro y acertó); el último debería tener un costo altísimo (el
modelo estaba casi seguro... y se equivocó por completo). Formalicemos esa
intuición.

Si tratamos $y$ como una variable de Bernoulli (una moneda cargada que sale 1
con probabilidad $p$), la probabilidad de observar la etiqueta real de una
observación es:

$$P(y \mid x) = p^y(1-p)^{1-y}$$

Esta fórmula es más simple de lo que parece — es solo una forma compacta de
escribir "si $y=1$ usa $p$, si $y=0$ usa $1-p$", gracias a que elevar algo a la
potencia 0 lo vuelve 1 y "apaga" ese factor:

- Si $y=1$: $p^1(1-p)^0 = p \cdot 1 = p$. Queremos que sea **alta** (el modelo
  asignó alta probabilidad a la clase positiva, que sí ocurrió).
- Si $y=0$: $p^0(1-p)^1 = 1 \cdot (1-p) = 1-p$. Queremos que sea **alta**
  (el modelo asignó baja probabilidad a la clase positiva, que no ocurrió).

Verifiquemos con los 4 casos de arriba que esta fórmula sí captura la intuición
que ya teníamos:

In [5]:
casos = casos.with_columns(
    (pl.col("p_predicha") ** pl.col("y_real") * (1 - pl.col("p_predicha")) ** (1 - pl.col("y_real"))).alias("P(y|x)")
)
casos

caso,y_real,p_predicha,P(y|x)
str,i64,f64,f64
"""muy seguro y acertó""",1,0.99,0.99
"""algo seguro y acertó""",1,0.6,0.6
"""algo seguro y falló""",1,0.4,0.4
"""muy seguro y falló""",1,0.01,0.01


Como esperábamos: `P(y|x)` es alta cuando el modelo acertó con confianza, y baja
cuando falló con confianza. Para $n$ observaciones independientes, la
probabilidad conjunta de acertar todas (la **verosimilitud**) es el producto de
esas probabilidades individuales:

$$L = \prod_{i=1}^{n} p_i^{y_i}(1-p_i)^{1-y_i}$$

## 3. Por qué usamos logaritmo: de productos a sumas

Multiplicar cientos de números entre 0 y 1 es incómodo por dos razones
prácticas: el resultado se vuelve absurdamente pequeño (riesgo de que la
computadora lo redondee a 0) y es difícil derivar un producto de muchos
términos. El logaritmo resuelve ambas cosas a la vez, porque tiene una
propiedad clave: convierte productos en sumas, $\log(a \cdot b) = \log(a) +
\log(b)$.

Veámoslo con números concretos antes de aplicarlo a la fórmula:

In [6]:
probabilidades = np.array([0.9, 0.85, 0.7, 0.95, 0.8])
producto_directo = np.prod(probabilidades)
suma_de_logs = np.sum(np.log(probabilidades))
print(f"Producto de las 5 probabilidades:        {producto_directo:.6f}")
print(f"exp(suma de sus logaritmos):              {np.exp(suma_de_logs):.6f}  <- mismo resultado")
print(f"Con solo 5 valores ya se hace pequeño; con miles de datos, el producto directo puede colapsar a 0.")

Producto de las 5 probabilidades:        0.406980
exp(suma de sus logaritmos):              0.406980  <- mismo resultado
Con solo 5 valores ya se hace pequeño; con miles de datos, el producto directo puede colapsar a 0.


Además, la forma de $\log(x)$ para $x$ entre 0 y 1 es justo lo que queremos
para medir "sorpresa": mientras más cerca de 0 esté una probabilidad, más
negativo (y más grande en magnitud) se vuelve su logaritmo.

In [7]:
x_log = np.linspace(0.01, 1, 200)
px.line(
    pl.DataFrame({"probabilidad": x_log, "log(probabilidad)": np.log(x_log)}),
    x="probabilidad", y="log(probabilidad)",
    title="log(x) crece muy despacio cerca de 1, pero cae en picada cerca de 0",
).show()

Tomamos entonces el logaritmo de la verosimilitud (que, por la propiedad de
arriba, convierte el producto en una suma):

$$\log L = \sum_{i=1}^{n} [y_i\log(p_i) + (1-y_i)\log(1-p_i)]$$

## 4. La función de costo: log loss

Los optimizadores de scikit-learn (y de la mayoría de librerías) están hechos
para **minimizar**, no para maximizar. Como maximizar $\log L$ es lo mismo que
minimizar $-\log L$, usamos el negativo del promedio de la log-verosimilitud
como función de costo:

$$J(w,b) = -\frac{1}{n} \sum_{i=1}^{n} [y_i\log(p_i) + (1-y_i)\log(1-p_i)]$$

Esta es la **entropía cruzada binaria** (o *log loss*). Fíjate que, para cada
observación, en realidad solo "sobrevive" uno de los dos términos —el otro se
apaga por el mismo mecanismo de potencias que vimos en la sección 2—, así que
el costo de una sola observación es:

$$\text{costo}_i = \begin{cases} -\log(p_i) & \text{si } y_i = 1 \\ -\log(1-p_i) & \text{si } y_i = 0 \end{cases}$$

Grafiquemos ambos casos por separado — literalmente son la curva de
$-\log(x)$ de la sección anterior, una vez sin voltear y otra volteada:

In [8]:
p = np.linspace(0.001, 0.999, 300)
costos = pl.DataFrame({
    "probabilidad_predicha": np.concatenate([p, p]),
    "costo": np.concatenate([-np.log(p), -np.log(1 - p)]),
    "etiqueta_real": ["y = 1"] * len(p) + ["y = 0"] * len(p),
})
fig = px.line(
    costos, x="probabilidad_predicha", y="costo", color="etiqueta_real",
    title="La log loss castiga poco los aciertos seguros y muchísimo los errores seguros",
)
fig.add_annotation(x=0.95, y=-np.log(0.95), text="y=1, p alta: costo casi 0", showarrow=True, arrowhead=2, ax=-60, ay=-40)
fig.add_annotation(x=0.05, y=-np.log(0.05), text="y=1, p baja: costo dispara", showarrow=True, arrowhead=2, ax=60, ay=-40)
fig.show()

Penaliza poco una predicción segura y correcta, pero penaliza muchísimo una
predicción segura y equivocada: si $y=1$ y $p$ se acerca a 0, entonces
$-\log(p)$ crece sin límite. Es exactamente el comportamiento que buscamos
cuando el modelo afirma una probabilidad casi imposible para el resultado que
sí ocurrió.

In [9]:
def binary_cross_entropy(y_true, y_prob):
    # Evita log(0) por redondeo de punto flotante.
    eps = np.finfo(float).eps
    p = np.clip(np.asarray(y_prob), eps, 1 - eps)
    y = np.asarray(y_true)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

casos = casos.with_columns(
    pl.struct(["y_real", "p_predicha"]).map_elements(
        lambda fila: binary_cross_entropy([fila["y_real"]], [fila["p_predicha"]]),
        return_dtype=pl.Float64,
    ).alias("costo (log loss)")
)
casos

caso,y_real,p_predicha,P(y|x),costo (log loss)
str,i64,f64,f64,f64
"""muy seguro y acertó""",1,0.99,0.99,0.01005
"""algo seguro y acertó""",1,0.6,0.6,0.510826
"""algo seguro y falló""",1,0.4,0.4,0.916291
"""muy seguro y falló""",1,0.01,0.01,4.60517


## 5. El gradiente: sorprendentemente simple

Para reducir el costo, un algoritmo de descenso de gradiente necesita la
derivada del costo respecto a los parámetros — la misma idea de "pendiente que
indica hacia dónde moverse" que se explica en detalle en
[`01_funcion_de_costo.ipynb` de regresión lineal](../../regresion/regresion_lineal/01_funcion_de_costo.ipynb).
Aquí, al derivar la pérdida respecto al puntaje $z$, todos los logaritmos y
exponenciales se cancelan y queda una expresión muy simple:

$$\frac{\partial J}{\partial z} = p - y$$

En palabras: el "empuje" para corregir los parámetros es, literalmente, **la
probabilidad predicha menos la etiqueta real**. Si el modelo predijo $p=0.9$
para una observación con $y=1$, el error es pequeño ($-0.1$); si predijo
$p=0.9$ para $y=0$, el error es grande ($0.9$). Veámoslo con los mismos 4 casos
de antes:

In [10]:
casos = casos.with_columns(((pl.col("p_predicha") - pl.col("y_real"))).alias("p - y (gradiente por observación)"))
fig = px.bar(
    casos, x="caso", y="p - y (gradiente por observación)",
    title="El gradiente por observación crece en magnitud cuanto más segura y equivocada fue la predicción",
    color="p - y (gradiente por observación)", color_continuous_scale="RdBu_r", range_color=[-1, 1],
)
fig.add_hline(y=0, line_color="black")
fig.show()

Por la regla de la cadena, para todo el conjunto de datos:

$$\nabla_w J = \frac{1}{n}X^T(p-y), \qquad \frac{\partial J}{\partial b} = \frac{1}{n}\sum_i(p_i-y_i)$$

La actualización por descenso de gradiente mueve los parámetros para reducir
ese error probabilístico. En la práctica, `scikit-learn` resuelve esta
optimización internamente.

## 6. Ideas clave

- La regresión logística modela **probabilidades**, no etiquetas directamente;
  la sigmoide es lo que garantiza que esa probabilidad quede siempre entre 0 y
  1, a diferencia de una recta común.
- Su costo procede de máxima verosimilitud para una variable Bernoulli: quiere
  maximizar la probabilidad de haber observado exactamente las etiquetas
  reales.
- Tomar logaritmo convierte un producto de muchas probabilidades pequeñas en
  una suma manejable, sin cambiar dónde está el óptimo.
- La entropía cruzada binaria evalúa la calidad de las probabilidades
  predichas: casi no penaliza los aciertos seguros, y penaliza sin límite los
  errores seguros.
- El gradiente respecto al puntaje es simplemente `p - y`: el tamaño del error
  de probabilidad.
- Durante una implementación numérica hay que evitar $\log(0)$; por eso se
  recortan las probabilidades o se usan implementaciones estables.
- Para evaluar un clasificador completo, complementa la log loss con métricas
  como precisión, *recall*, F1, ROC-AUC y una matriz de confusión (temas que
  verás en un notebook dedicado a métricas de clasificación).

**Ejercicio:** cambia las cuatro probabilidades de `casos` (sección 2) y vuelve
a ejecutar el notebook. Antes de mirar el resultado, predice: ¿qué le pasa al
costo cuando una observación positiva recibe $p=0.001$? Luego crea un ejemplo
con $y=0$ y compara cómo cambia el signo del gradiente `p - y`.